# ReefScan AI — Colab GPU training (E4 + E5 + E6)

Runs the **fish species** experiment matrix on fishnet.ai crops (feature-extraction screen → fine-tune the winner) + XAI/enhancement benchmarks.

**Before Run All:** Runtime → *Change runtime type* → **T4 GPU**. Duration ≈ 2.5–3.5 h. If *Allocating runtime* hangs, retry later — at the end the artifact zip downloads straight to your PC.

In [ ]:
# --- 0. Clone the repo + install dependencies ------------------------------
!git clone https://github.com/Jayant1711/reefscan.git /content/reefscan
%cd /content/reefscan
!pip install -q -r requirements.txt


In [ ]:
# --- 1. GPU + test-suite check (fails loudly if anything is broken) --------
import torch, sys
assert torch.cuda.is_available(), "NO GPU — enable the accelerator and re-run!"
print("GPU:", torch.cuda.get_device_name(0))
!python -m pytest tests/ -q


In [ ]:
# --- 2. FishNet v0.1.2 from fishnet.ai S3 (5.7 GB) -> bbox crops ----------
!python download_data.py --dataset fishnet
!python download_data.py --prepare-fishnet
import glob
print("fish crops:", len(glob.glob("/content/reefscan_data/fishnet/curated/*/*.jpg")))

In [ ]:
# --- 3. Data cleaning (corrupt/dup/leakage scan + quarantine) --------------
!python clean_data.py --dataset fishnet --apply --platform colab

In [ ]:
# --- 4. E4 — fish feature-extraction screen (all 4 backbones, ~1.5 h) ------
!python train.py --task fish --model all --strategy feature_extraction --platform colab
import pandas as pd
pd.read_csv("results/fish/model_leaderboard.csv")

In [ ]:
# --- 5. E5 — auto-pick the winner and fine-tune it (~1.5 h) -----------------
import pandas as pd, subprocess, sys
lb = pd.read_csv("results/fish/model_leaderboard.csv")
fe = lb[lb["strategy"] == "feature_extraction"]
winner = fe.nlargest(1, "weighted_f1")["model"].iloc[0]
print("Fine-tuning FE-screen winner:", winner)
subprocess.run([sys.executable, "train.py", "--task", "fish", "--model", winner,
                "--strategy", "finetune", "--platform", "colab"], check=True)
!python evaluate.py --task fish
pd.read_csv("results/fish/model_leaderboard.csv")

In [ ]:
# --- 6. E6 — XAI grid + enhancement benchmark on real data ------------------
import glob
sample = (glob.glob("/content/reefscan_data/fishnet/curated/test/*/*.jpg") or
          glob.glob("/content/reefscan_data/fishnet/curated/train/*/*.jpg"))[0]
!python evaluate.py --enhancement --data /content/reefscan_data/fishnet/curated/train
!python evaluate.py --xai --image {sample} || true

In [ ]:
# --- 9. Save everything: ledger + artifact zip + BROWSER DOWNLOAD ----------
!python scripts/log_experiment.py
!python scripts/export_artifacts.py
import glob
from google.colab import files
zip_path = sorted(glob.glob("reefscan_artifacts_colab_*.zip"))[-1]
files.download(zip_path)   # saves the trained models to your PC
